It parses each PDF with PyMuPDF (column-aware, so two-column papers don't come out scrambled), saves the full text, and prints a compact report you can paste back to me.

In [1]:
!pip -q install pymupdf

import fitz, re, os, glob
from google.colab import files  # remove this line + the upload cell if running locally

# --- 1. upload your 4-5 PDFs ---
os.makedirs("papers", exist_ok=True)
os.makedirs("papers_txt", exist_ok=True)
for name, data in files.upload().items():
    open(f"papers/{name}", "wb").write(data)

# --- 2. column-aware extraction ---
def page_text(page):
    W = page.rect.width
    mid = W / 2
    blocks = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]
    blocks.sort(key=lambda b: b[1])

    narrow = [b for b in blocks if (b[2] - b[0]) < 0.55 * W]
    left_n = sum(1 for b in narrow if (b[0] + b[2]) / 2 < mid)
    right_n = len(narrow) - left_n
    two_col = left_n >= 3 and right_n >= 3

    if not two_col:
        return "\n".join(b[4].strip() for b in blocks), False

    out, band = [], []
    def flush():
        left = sorted([b for b in band if (b[0] + b[2]) / 2 < mid], key=lambda b: b[1])
        right = sorted([b for b in band if (b[0] + b[2]) / 2 >= mid], key=lambda b: b[1])
        out.extend(left + right)
        band.clear()

    for b in blocks:
        if b[0] < mid < b[2] and (b[2] - b[0]) > 0.6 * W:  # full-width block (title, wide figure/table)
            flush()
            out.append(b)
        else:
            band.append(b)
    flush()
    return "\n".join(b[4].strip() for b in out), True

# --- 3. run + diagnostics ---
for path in sorted(glob.glob("papers/*.pdf")):
    doc = fitz.open(path)
    pages, two_col_pages = [], 0
    for p in doc:
        t, tc = page_text(p)
        pages.append(t)
        two_col_pages += tc
    full = "\n\n".join(f"[[PAGE {i+1}]]\n{t}" for i, t in enumerate(pages))
    clean = "\n\n".join(pages)
    base = os.path.basename(path)
    open(f"papers_txt/{base}.txt", "w").write(full)

    refs = [m.start() for m in re.finditer(r"\n\s*(references|bibliography)\s*\n", clean, re.I)]
    ref_pos = f"{100*refs[-1]/len(clean):.0f}% into doc" if refs else "NOT FOUND"
    heads = sorted(set(m.group(0).strip().lower() for m in re.finditer(
        r"^\s*(?:\d+\.?\s*)?(limitations?|future work|conclusions?|discussion)\s*$", clean, re.I | re.M)))

    print("=" * 70)
    print(base)
    print(f"pages={len(doc)} | chars={len(clean)} | chars/page={len(clean)//max(len(doc),1)} "
          f"| two-column pages={two_col_pages}/{len(doc)}")
    print(f"hyphen line-breaks={len(re.findall(r'\w-\n\w', clean))} | "
          f"bad chars(\\ufffd)={clean.count(chr(0xfffd))} | (cid:) tokens={clean.count('(cid:')} | "
          f"ligatures={sum(clean.count(c) for c in 'ﬁﬂﬀﬃﬄ')}")
    print(f"references heading: {ref_pos} | headings found: {heads or 'none'}")
    print("--- page 1 (first 700 chars) ---")
    print(pages[0][:700])
    mid_i = len(pages) // 2
    print(f"--- page {mid_i+1} (first 500 chars) ---")
    print(pages[mid_i][:500])
    if refs:
        print("--- around references heading (300 chars) ---")
        print(clean[max(refs[-1]-150, 0): refs[-1]+150])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.6 MB/s eta 0:00:00


Saving A_machine_learning_based_depression_screening_fram.pdf to A_machine_learning_based_depression_screening_fram.pdf
Saving A_neuro-fuzzy_approach.pdf to A_neuro-fuzzy_approach.pdf
Saving AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf to AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf
Saving An_in-depth_analysis_of_machine.pdf to An_in-depth_analysis_of_machine.pdf
Saving Predicting_depression_level_based.pdf to Predicting_depression_level_based.pdf
AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf
pages=8 | chars=21569 | chars/page=2696 | two-column pages=4/8
hyphen line-breaks=4 | bad chars(\ufffd)=0 | (cid:) tokens=0 | ligatures=0
references heading: 82% into doc | headings found: ['6. conclusion']
--- page 1 (first 700 chars) ---
See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/357935204
A Genetic Neuro-Fuzzy System for Diagnosing Clinical Depression
Research · January 2021
CITATIONS
2
3 authors, including:
Ade

If body text starts vanishing near page edges, adjust the 0.07 and 0.93 cutoffs. I kept the page markers so you can cite page numbers later. Once this runs clean, the next step is chunking with paper and section tags.

In [3]:
!pip -q install pymupdf

import fitz, re, os, glob, unicodedata

# --- upload PDFs only if they're not already in the Colab session ---
os.makedirs("papers", exist_ok=True)
os.makedirs("papers_txt", exist_ok=True)
if not glob.glob("papers/*.pdf"):
    from google.colab import files
    for name, data in files.upload().items():
        open(f"papers/{name}", "wb").write(data)

SKIP_PAGES = {  # 1-indexed: cover / declaration pages
    "AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf": {1},
    "Predicting_depression_level_based.pdf": {1, 2, 19},
}

def clean(text):
    text = unicodedata.normalize("NFKC", text)           # ﬁ -> fi, math italics -> plain
    text = re.sub(r"[\uac00-\ud7a3]", "", text)          # Hangul junk from math glyphs
    text = re.sub(r"(?<=[a-z])-\n(?=[a-z])", "", text)   # join hyphenated line breaks
    return re.sub(r"[ \t]+\n", "\n", text)

def page_text(page):
    W, H = page.rect.width, page.rect.height
    mid = W / 2
    # drop text blocks in the top 7% / bottom 7% of the page (running headers/footers)
    blocks = [b for b in page.get_text("blocks")
              if b[6] == 0 and b[4].strip() and b[1] > 0.07 * H and b[3] < 0.93 * H]
    blocks.sort(key=lambda b: b[1])

    narrow = [b for b in blocks if (b[2] - b[0]) < 0.55 * W]
    left_n = sum(1 for b in narrow if (b[0] + b[2]) / 2 < mid)
    right_n = len(narrow) - left_n
    two_col = left_n >= 3 and right_n >= 3

    if not two_col:
        return "\n".join(b[4].strip() for b in blocks), False

    out, band = [], []
    def flush():
        left = sorted([b for b in band if (b[0] + b[2]) / 2 < mid], key=lambda b: b[1])
        right = sorted([b for b in band if (b[0] + b[2]) / 2 >= mid], key=lambda b: b[1])
        out.extend(left + right)
        band.clear()

    for b in blocks:
        if b[0] < mid < b[2] and (b[2] - b[0]) > 0.6 * W:   # full-width block
            flush()
            out.append(b)
        else:
            band.append(b)
    flush()
    return "\n".join(b[4].strip() for b in out), True

# --- run + save + stats ---
for path in sorted(glob.glob("papers/*.pdf")):
    base = os.path.basename(path)
    doc = fitz.open(path)
    pages = []
    for i, p in enumerate(doc, 1):
        if i in SKIP_PAGES.get(base, set()):
            continue
        t, _ = page_text(p)
        t = clean(t)
        if len(t) > 200:                                  # drops figure-only pages
            pages.append((i, t))

    full = "\n\n".join(f"[[PAGE {n}]]\n{t}" for n, t in pages)
    open(f"papers_txt/{base}.txt", "w").write(full)
    body = "\n".join(t for _, t in pages)

    print("=" * 70)
    print(base)
    print(f"pages kept={len(pages)}/{len(doc)} | chars={len(body)} | chars/page={len(body)//max(len(pages),1)}")
    print(f"hyphen breaks left={len(re.findall(r'\w-\n\w', body))} | "
          f"ligatures left={sum(body.count(c) for c in 'ﬁﬂﬀﬃﬄ')} | "
          f"hangul left={len(re.findall(r'[\uac00-\ud7a3]', body))} | "
          f"'Journal Pre-proof' lines={body.count('Journal Pre-proof')} | "
          f"'PLOS ONE' lines={body.count('PLOS ONE')}")
    n, t = pages[len(pages) // 2]
    print(f"--- page {n} (first 400 chars) ---")
    print(t[:400])

AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf
pages kept=5/8 | chars=19768 | chars/page=3953
hyphen breaks left=2 | ligatures left=0 | hangul left=0 | 'Journal Pre-proof' lines=0 | 'PLOS ONE' lines=0
--- page 4 (first 400 chars) ---
data was used to design the system using Java.
represents the diagnostic outcome. A total of 134 dataset
obtained were examined by us and some experienced
psychiatrists at the Medical center in arriving at the
diagnostic prediction. At Federal Medical Center Owo, they
employed the Becks Depression Inventory in diagnosing
depressive episodes. The beck inventory scale was model
into a mathematical e
A_machine_learning_based_depression_screening_fram.pdf
pages kept=28/29 | chars=89673 | chars/page=3202
hyphen breaks left=3 | ligatures left=0 | hangul left=0 | 'Journal Pre-proof' lines=0 | 'PLOS ONE' lines=0
--- page 16 (first 400 chars) ---
Fig 4. Volin plot for the selected common features obtained from the EEG data of the three electrodes i.e., Fp1, FpZ

The Genetic paper's column-order check should print True. The 11 to 13 leftover hyphens are real compounds like "non-\nlinear", so ignore them. Two other known leftovers are also fine:

Equations stay garbled ("lxðAÞ 1⁄4 ..."), and Saha's equation pages lose their variables. That doesn't matter for gap questions.
The PLOS reference pages come out in a scrambled order, but we cut everything after "References" anyway.

In [4]:
!pip -q install pymupdf
try:
    import pymupdf as fitz
except ImportError:
    import fitz
import re, os, glob, unicodedata

os.makedirs("papers", exist_ok=True)
os.makedirs("papers_txt", exist_ok=True)
if not glob.glob("papers/*.pdf"):          # skips the upload if PDFs are already there
    from google.colab import files
    for name, data in files.upload().items():
        open(f"papers/{name}", "wb").write(data)

SKIP_PAGES = {  # 1-indexed: cover / declaration pages
    "AGeneticNeuro-FuzzySystemforDiagnosingClinical.pdf": {1},
    "Predicting_depression_level_based.pdf": {1, 2, 19},
}

def clean(text):
    text = unicodedata.normalize("NFKC", text)                        # ﬁ -> fi
    text = re.sub(r"[\uac00-\ud7a3]", "", text)                       # Hangul junk from math glyphs
    text = re.sub(r"[ \t]+\n", "\n", text)                            # strip trailing spaces FIRST
    text = re.sub(r"(?<=[a-z])-\n(?=[a-z])", "", text)                # then join hyphenated breaks
    text = re.sub(r"(?m)^\s*Journal Pre-proof\s*$\n?", "", text)      # mid-page watermark
    text = re.sub(r"(?m)^https://doi\.org/10\.1371/\S+\s*$\n?", "", text)  # PLOS figure/table DOI lines
    return text

def get_lines(page):
    H = page.rect.height
    lines = []
    for b in page.get_text("dict")["blocks"]:
        if b["type"] != 0:
            continue
        for l in b["lines"]:
            txt = "".join(s["text"] for s in l["spans"]).strip()
            x0, y0, x1, y1 = l["bbox"]
            if txt and y0 > 0.07 * H and y1 < 0.93 * H:      # drop running headers/footers
                lines.append((x0, y0, x1, y1, txt))
    return lines

def page_text(page):
    W, H = page.rect.width, page.rect.height
    mid = W / 2
    lines = get_lines(page)
    narrow = [l for l in lines if (l[2] - l[0]) < 0.55 * W]
    L = sum(1 for l in narrow if (l[0] + l[2]) / 2 < mid)
    R = len(narrow) - L

    if not (L >= 10 and R >= 10):                            # single column: keep block order
        blocks = [b for b in page.get_text("blocks")
                  if b[6] == 0 and b[4].strip() and b[1] > 0.07 * H and b[3] < 0.93 * H]
        blocks.sort(key=lambda b: b[1])
        return "\n".join(b[4].strip() for b in blocks), False

    lines.sort(key=lambda l: l[1])                           # two columns: line by line
    out, band = [], []
    def flush():
        out.extend(sorted([l for l in band if (l[0] + l[2]) / 2 < mid], key=lambda l: l[1]))
        out.extend(sorted([l for l in band if (l[0] + l[2]) / 2 >= mid], key=lambda l: l[1]))
        band.clear()
    for l in lines:
        if l[0] < mid < l[2] and (l[2] - l[0]) > 0.6 * W:    # full-width line
            flush(); out.append(l)
        else:
            band.append(l)
    flush()
    return "\n".join(l[4] for l in out), True

for path in sorted(glob.glob("papers/*.pdf")):
    base = os.path.basename(path)
    doc = fitz.open(path)
    pages = []
    for i, p in enumerate(doc, 1):
        if i in SKIP_PAGES.get(base, set()):
            continue
        t, _ = page_text(p)
        t = clean(t)
        if len(t) > 200:                                     # drops figure-only pages
            pages.append((i, t))

    open(f"papers_txt/{base}.txt", "w").write(
        "\n\n".join(f"[[PAGE {n}]]\n{t}" for n, t in pages))
    body = "\n".join(t for _, t in pages)
    print(f"{base[:45]:45} kept={len(pages)}/{len(doc)} chars={len(body)} "
          f"hyphens_left={len(re.findall(r'\w-\n\w', body))} "
          f"ligatures={sum(body.count(c) for c in 'ﬁﬂﬀﬃﬄ')} "
          f"watermark={body.count('Journal Pre-proof')}")
    if base.startswith("AGenetic"):
        d = dict(pages)
        print("   Genetic p4 column order ok:",
              d[4].find("Data collection") < d[4].find("represents the diagnostic"))

AGeneticNeuro-FuzzySystemforDiagnosingClinica kept=5/8 chars=19912 hyphens_left=2 ligatures=0 watermark=0
   Genetic p4 column order ok: True
A_machine_learning_based_depression_screening kept=28/29 chars=89080 hyphens_left=5 ligatures=0 watermark=0
A_neuro-fuzzy_approach.pdf                    kept=9/9 chars=38571 hyphens_left=13 ligatures=0 watermark=0
An_in-depth_analysis_of_machine.pdf           kept=12/12 chars=62499 hyphens_left=11 ligatures=0 watermark=0
Predicting_depression_level_based.pdf         kept=16/19 chars=64208 hyphens_left=11 ligatures=0 watermark=0


In [1]:
!pip -q install langchain-text-splitters langchain-core

import re, os, glob, json, bisect, collections
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

PAPERS = {  # file stem -> short label used in citations (edit freely)
    "AGeneticNeuro-FuzzySystemforDiagnosingClinical": "Adegboye2021 (Genetic Neuro-Fuzzy)",
    "A_machine_learning_based_depression_screening_fram": "Khan2024 (EEG temporal features + ML)",
    "A_neuro-fuzzy_approach": "Chattopadhyay2017 (Neuro-fuzzy diagnosis)",
    "An_in-depth_analysis_of_machine": "Zulfiker2021 (ML + feature selection)",
    "Predicting_depression_level_based": "Saha2024 (Fuzzy logic depression level)",
}

SECTION_RULES = [   # checked in order; matched against the start of a short heading line
    ("back_matter", ("acknowledg", "author contributions", "supporting information", "data availability",
                     "code availability", "availability of data", "funding", "declaration of",
                     "competing interest", "conflict of interest")),
    ("abstract",     ("abstract",)),
    ("introduction", ("introduction",)),
    ("related_work", ("related work", "review of related", "background", "literature", "current literature")),
    ("methods",      ("method", "materials and method", "proposed")),
    ("results",      ("result", "experimental result", "experiment", "evaluation", "system testing")),
    ("discussion",   ("discussion", "findings", "implications", "theoretical", "managerial")),
    ("conclusion",   ("conclusion", "limitation", "future work", "future scope")),
]

def classify(line):
    """Return canonical section name if `line` looks like a section heading, else None."""
    s = line.strip()
    norm = re.sub(r"^\s*\d+(\.\d+)*\.?\s*", "", s.lower())
    if re.sub(r"\s+", "", norm).startswith("abstract"):                 # also "a b s t r a c t"
        return "abstract"
    if norm.startswith("future work") and ":" in norm[:14]:            # inline "Future work: ..."
        return "conclusion"
    body = re.sub(r"^\s*\d+(\.\d+)*\.?\s*", "", s)
    if not body or not body[0].isupper():                              # headings start with a capital
        return None
    if len(s) > 70 or len(body.split()) > 8 or re.search(r"[\d(),]", body) or s.endswith((".", ";")):
        return None                                                    # body text / table cells, not headings
    for name, prefixes in SECTION_RULES:
        if norm.startswith(prefixes):
            return name
    return None

# Saha's first page has an odd 2-column layout: its abstract text lands after the intro.
ABSTRACT_SPAN = {"Predicting_depression_level_based": ("Millions of individuals die", "distinct degrees")}

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, add_start_index=True)
docs, coverage = [], collections.defaultdict(collections.Counter)

for path in sorted(glob.glob("papers_txt/*.txt")):
    stem = os.path.basename(path).replace(".pdf.txt", "")
    if stem not in PAPERS:
        continue
    label = PAPERS[stem]
    raw = open(path).read()
    parts = re.split(r"\[\[PAGE (\d+)\]\]\n", raw)[1:]
    lines = [(int(n), l.strip()) for n, txt in zip(parts[0::2], parts[1::2])
             for l in txt.split("\n") if l.strip()]

    # cut the reference list (last line that is just "References")
    ref_idx = [i for i, (_, l) in enumerate(lines) if re.fullmatch(r"(?i)references?", l)]
    if ref_idx:
        lines = lines[:ref_idx[-1]]

    # group consecutive lines into (section, text, page-offsets) runs
    runs, cur, resume = [], "front", None
    span = ABSTRACT_SPAN.get(stem)
    for page, line in lines:
        if span and resume is None and span[0] in line:
            resume, cur = cur, "abstract"
        sec = None if (span and resume is not None) else classify(line)
        if sec and not line.lower().lstrip("0123456789. ").startswith("abstract"):
            cur = sec                                   # pure heading line: switch section, don't index it
            continue
        if sec == "abstract":
            cur = "abstract"
        if not runs or runs[-1]["section"] != cur:
            runs.append({"section": cur, "text": "", "starts": [], "pages": []})
        r = runs[-1]
        r["starts"].append(len(r["text"])); r["pages"].append(page)
        r["text"] += line + " "
        if span and resume is not None and span[1] in line:
            cur, resume = resume, None

    for r in runs:
        if r["section"] in ("front", "back_matter"):
            continue
        for d in splitter.create_documents([r["text"]], [{}]):
            text = d.page_content.strip()
            if len(text) < 150:
                continue
            page = r["pages"][bisect.bisect_right(r["starts"], d.metadata["start_index"]) - 1]
            digit_ratio = sum(c.isdigit() for c in text) / len(text)
            docs.append(Document(page_content=text, metadata={
                "paper": label, "section": r["section"], "page": page,
                "table_like": digit_ratio > 0.25}))
            coverage[label][r["section"]] += 1

for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i

print(f"total chunks: {len(docs)}\n")
need = ["abstract", "introduction", "methods", "results", "conclusion"]
for label, cnt in coverage.items():
    missing = [s for s in need if cnt[s] == 0]
    print(f"{label}\n   {dict(cnt)}" + (f"\n   MISSING: {missing}" if missing else ""))

sample = next(d for d in docs if d.metadata["section"] == "conclusion")
print("\n--- sample conclusion chunk ---")
print(sample.metadata); print(sample.page_content[:500])

json.dump([{"text": d.page_content, "metadata": d.metadata} for d in docs],
          open("chunks.json", "w"), indent=1)

total chunks: 270

Adegboye2021 (Genetic Neuro-Fuzzy)
   {'abstract': 3, 'introduction': 5, 'related_work': 3, 'methods': 4, 'results': 2, 'discussion': 2, 'conclusion': 1}
Khan2024 (EEG temporal features + ML)
   {'abstract': 3, 'introduction': 2, 'related_work': 8, 'methods': 28, 'results': 31, 'conclusion': 2}
Chattopadhyay2017 (Neuro-fuzzy diagnosis)
   {'abstract': 2, 'introduction': 7, 'related_work': 7, 'methods': 11, 'results': 10, 'discussion': 3, 'conclusion': 1}
Zulfiker2021 (ML + feature selection)
   {'abstract': 2, 'introduction': 7, 'related_work': 8, 'methods': 29, 'results': 18, 'conclusion': 3}
Saha2024 (Fuzzy logic depression level)
   {'introduction': 7, 'abstract': 2, 'related_work': 9, 'methods': 27, 'results': 9, 'discussion': 11, 'conclusion': 3}

--- sample conclusion chunk ---
{'paper': 'Adegboye2021 (Genetic Neuro-Fuzzy)', 'section': 'conclusion', 'page': 7, 'table_like': False, 'chunk_id': 19}
As can be seen, a large amount of work has already been done in t